In [ ]:
import os
import time
import math
import pandas as pd
import numpy as np
import pylab as plt
import seaborn as sns

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black

jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

In [ ]:
VERSION = "research-article_aimrd_f"
BASELINE = "baseline_2026-01-23"
RESULTS_PATH = os.path.join("../data/results/", BASELINE, VERSION)
selection = ["delve", "crucial", "potential", "these", "significant", "important"]
start_date = "11-2017"
end_date = "12-2025"

os.makedirs(os.path.join("../results/plots", VERSION), exist_ok=True)

months = np.arange(1, 13)
years = np.arange(2000, 2025)

### Single words

In [ ]:
frequency_dfs = utils.load_freqs(RESULTS_PATH, selection, start_date)

In [ ]:
frequency_dfs["potential"]

In [ ]:
freq_dfs_agg = {}

for word, df in frequency_dfs.items():
    df = df.copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "section"]).mean().reset_index()
    df["section"] = pd.Categorical(
        df["section"],
        categories=[
            "abstract",
            "introduction",
            "methods",
            "results",
            "discussion",
            "full",
        ],
        ordered=True,
    )

    freq_dfs_agg[word] = df

In [ ]:
freq_dfs_agg["potential"]["section"].cat.categories

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(7, 4.5), layout="constrained")

for i, word in enumerate(selection):
    ax = axs.flat[i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(
        data=frequency_dfs[word], x="time", y="frequency", hue="section", ax=ax
    )
    sns.lineplot(
        data=frequency_dfs[word],
        x="time",
        y="projection",
        hue="section",
        ax=ax,
        alpha=0.5,
        linestyle="-.",
        legend=False,
    )
    ax.set_title(word)
    ax.set_xlabel(None)
    if i < 3:
        ax.set_xticks([2021, 2022, 2023, 2024, 2025])
        ax.set_xticklabels([])
    else:
        ax.set_xticks([2021, 2022, 2023, 2024, 2025])
        ax.set_xticklabels([2021, 2022, 2023, 2024, 2025], rotation=60)
    if i not in [0, 3]:
        ax.set_ylabel(None)
    if not i == 0:
        ax.get_legend().set_visible(False)

plt.savefig(os.path.join("../results/plots", VERSION, "word_freqs_time.png"), dpi=300)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(10, 4), layout="constrained")

for i, word in enumerate(selection):

    # plot diffs
    ax = axs[0, i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(data=frequency_dfs[word], x="time", y="diff", hue="section", ax=ax)

    # plot ratios
    ax = axs[1, i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(data=frequency_dfs[word], x="time", y="ratio", hue="section", ax=ax)

    for j in range(2):
        ax = axs[j, i]
        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(word)
        if j == 1:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([2021, 2022, 2023, 2024, 2025], rotation=60)
        else:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 0 and j == 0):
            ax.get_legend().set_visible(False)

plt.savefig(os.path.join("../results/plots", VERSION, "word_diff_ratio.png"), dpi=300)

is there a way to standardize between sections? bc the longer a section is, the more likely it is that a word occurs? or give mean word counts per sections?

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(8, 4), layout="constrained")

for i, word in enumerate(selection):

    # plot diffs
    ax = axs[0, i]
    sns.pointplot(data=freq_dfs_agg[word], x="section", y="diff", hue="time", ax=ax)

    # plot ratios
    ax = axs[1, i]
    sns.pointplot(data=freq_dfs_agg[word], x="section", y="ratio", hue="time", ax=ax)

    for j in range(2):
        ax = axs[j, i]
        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(word)
        if j == 1:
            ax.set_xticklabels(
                freq_dfs_agg["potential"]["section"].cat.categories, rotation=60
            )
        else:
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 0 and j == 0):
            ax.get_legend().set_visible(False)

plt.savefig(
    os.path.join("../results/plots", VERSION, "word_freqs_section.png"), dpi=300
)

### Word groups

In [ ]:
group_frequency_dfs = utils.load_freqs(
    data_path=RESULTS_PATH,
    group_prefix="group_",
    start_date=start_date,
    end_date=end_date,
)

In [ ]:
group_frequency_dfs["common_words"]

In [ ]:
group_freq_dfs_agg = {}

for word, df in group_frequency_dfs.items():
    df = df.copy()
    df["time"] = list(map(math.floor, df["time"]))
    df = df.groupby(["time", "section"]).mean().reset_index()
    df["section"] = pd.Categorical(
        df["section"],
        categories=[
            "abstract",
            "introduction",
            "methods",
            "results",
            "discussion",
            "full",
        ],
        ordered=True,
    )

    group_freq_dfs_agg[word] = df

In [ ]:
group_freq_dfs_agg["common_words"]["section"].cat.categories

In [ ]:
#### Old version

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(5, 5), layout="constrained")

for j, var in enumerate(["frequency", "diff", "ratio"]):
    for i, group in enumerate(group_frequency_dfs.keys()):
        ax = axs[j, i]
        ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
        sns.lineplot(
            data=group_frequency_dfs[group],
            x="time",
            y=var,
            hue="section",
            ax=ax,
        )
        if var == "frequency":
            sns.lineplot(
                data=group_frequency_dfs[group],
                x="time",
                y="projection",
                hue="section",
                ax=ax,
                alpha=0.5,
                linestyle="-.",
                legend=False,
            )

        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(group)
            ax.set_ylim(0.15, 1.05)
        if j == 1:
            ax.set_ylim(-0.025, 0.25)
        if j == 2:
            ax.set_ylim(0.96, 1.7)
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([2021, 2022, 2023, 2024, 2025], rotation=60)
        else:
            ax.set_xticks([2021, 2022, 2023, 2024, 2025])
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 1 and j == 2):
            ax.get_legend().set_visible(False)

        axs[1, 0].axhline(
            y=0.22, color="black", linestyle="dotted", alpha=0.8
        )  # common
        axs[1, 1].axhline(y=0.18, color="black", linestyle="dotted", alpha=0.8)  # rare

plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "group_freqs_time.png",
    ),
    dpi=300,
)

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(7, 5), layout="constrained")

for j, var in enumerate(["frequency", "usage", "diff"]):
    for i, group in enumerate(group_frequency_dfs.keys()):
        ax = axs[j, i]
        ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
        sns.lineplot(
            data=group_frequency_dfs[group],
            x="time",
            y=var,
            hue="section",
            ax=ax,
        )
        if var == "frequency":
            sns.lineplot(
                data=group_frequency_dfs[group],
                x="time",
                y="projection",
                hue="section",
                ax=ax,
                alpha=0.5,
                linestyle="-.",
                legend=False,
            )

        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(group)
            ax.set_ylim(0.15, 1.05)
        if j == 1:
            ax.set_ylim(-0.2, 1.05)
        if j == 2:
            ax.set_ylim(-0.05, 1.05)
            ax.set_xticks(range(2018, 2026))
            ax.set_xticklabels(range(2018, 2026), rotation=60)
        else:
            ax.set_xticks(range(2018, 2026))
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 1 and j == 2):
            ax.get_legend().set_visible(False)


plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "group_freqs_time_old_version.png",
    ),
    dpi=300,
)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(4, 4), layout="constrained")

for i, var in enumerate(["frequency", "usage"]):

    ax = axs[i]
    ax.axvline(x=2022 + (11 / 12), linestyle="--", color="black", alpha=0.3)
    sns.lineplot(
        data=group_frequency_dfs["rare_words"],
        x="time",
        y=var,
        hue="section",
        ax=ax,
    )
    if var == "frequency":
        sns.lineplot(
            data=group_frequency_dfs["rare_words"],
            x="time",
            y="projection",
            hue="section",
            ax=ax,
            alpha=0.5,
            linestyle="-.",
            legend=False,
        )

    ax.set_xlabel(None)
    if i == 0:
        ax.set_title(group)
        ax.set_ylim(0.7, 1.05)
    if i == 1:
        ax.set_ylim(-0.2, 1.05)

    else:
        ax.set_xticks(range(2018, 2026))
        ax.set_xticklabels([])
    if not i == 0:
        ax.set_ylabel(None)
    if not (i == 1 and j == 2):
        ax.get_legend().set_visible(False)


plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "group_freqs_time.png",
    ),
)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(4, 4), layout="constrained")

for i, group in enumerate(group_freq_dfs_agg.keys()):

    # plot diffs
    ax = axs[0, i]
    sns.pointplot(
        data=group_freq_dfs_agg[group], x="section", y="diff", hue="time", ax=ax
    )

    # plot ratios
    ax = axs[1, i]
    sns.pointplot(
        data=group_freq_dfs_agg[group], x="section", y="ratio", hue="time", ax=ax
    )

    for j in range(2):
        ax = axs[j, i]
        ax.set_xlabel(None)
        if j == 0:
            ax.set_title(group)
            ax.set_ylim(-0.025, 0.25)
        if j == 1:
            ax.set_ylim(0.96, 1.7)
            ax.set_xticklabels(
                group_freq_dfs_agg["common_words"]["section"].cat.categories,
                rotation=60,
            )
        else:
            ax.set_xticklabels([])
        if not i == 0:
            ax.set_ylabel(None)
        if not (i == 1 and j == 1):
            ax.get_legend().set_visible(False)

        axs[0, 0].axhline(
            y=0.22, color="black", linestyle="dotted", alpha=0.8
        )  # common
        axs[0, 1].axhline(y=0.18, color="black", linestyle="dotted", alpha=0.8)  # rare

plt.savefig(
    os.path.join("../results/plots", VERSION, "group_freqs_section.png"), dpi=300
)